# Week 54, Day 5: Learning curves: how much data is enough?

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w054/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: How decoding error falls with trials

Complete decoder_fit (ridge with an unpenalised intercept on standardised counts) and, for each n, average the training and validation mean squared error over 10 repetitions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(545)
N = 20
centres = np.linspace(-1.2, 1.2, N)

def simulate(n, g):
    s = g.uniform(-1, 1, n)
    rates = 2 + 8 * np.exp(-0.5 * ((s[:, None] - centres) / 0.5) ** 2)
    return s, g.poisson(rates * 0.25)

def decoder_fit(X, s, lam=1.0):
    mu, sd = X.mean(0), X.std(0) + 1e-6
    A = np.c_[(X - mu) / sd, np.ones(len(s))]
    D = np.eye(A.shape[1]); D[-1, -1] = 0
    w = ...
    return lambda Xn: np.c_[(Xn - mu) / sd, np.ones(len(Xn))] @ w

ns = np.array([25, 50, 100, 200, 400, 800, 1600])
s_va, X_va = simulate(4000, r)
tr_err, va_err = np.zeros(len(ns)), np.zeros(len(ns))
for k, n in enumerate(ns):
    tr, va = [], []
    for rep in range(10):
        s_tr, X_tr = simulate(n, r)
        pred = decoder_fit(X_tr, s_tr)
        tr.append(...)
        va.append(...)
    tr_err[k], va_err[k] = np.mean(tr), np.mean(va)
    print(f"n = {n:4d}: training MSE {tr_err[k]:.4f}, validation MSE {va_err[k]:.4f}")
plt.loglog(ns, tr_err, "o-", label="training"); plt.loglog(ns, va_err, "o-", label="validation"); plt.xlabel("training trials"); plt.ylabel("MSE"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_counts():
    "simulate returns stimuli and counts of the right shapes"
    s_, X_ = simulate(10, np.random.default_rng(0))
    assert s_.shape == (10,) and X_.shape == (10, N)
def test_curves():
    "training error rises and validation error falls with n"
    assert tr_err[-1] > tr_err[0] and va_err[-1] < va_err[0]
def test_meet():
    "at 1,600 trials the gap between the curves is under a tenth of the validation error"
    assert (va_err[-1] - tr_err[-1]) < 0.1 * va_err[-1]

test_counts()
test_curves()
test_meet()
print("All checks passed")

### Exercise: Fitting and testing a power law

Fit E(n) = a n^(-b) + c to the validation errors for the first five sizes (25 to 400 trials) with scipy.optimize.curve_fit, then compare the extrapolation to 800 and 1,600 trials with the measured errors.

In [ ]:
from scipy.optimize import curve_fit

def power(n, a, b, c):
    return ...

params, _ = curve_fit(power, ns[:5], va_err[:5], p0=[1.0, 0.5, 0.01], bounds=([0, 0, 0], [100, 3, 1]))
a, b, c = params
print(f"fit: E(n) = {a:.2f} n^(-{b:.2f}) + {c:.4f}")
for n_, e in zip(ns[5:], va_err[5:]):
    print(f"n = {n_}: predicted {power(n_, *params):.4f}, measured {e:.4f}")
print(f"trials needed to halve the excess error from n = 400: about {400 * 2 ** (1 / b):.0f}")

In [ ]:
import numpy as np
def test_fit():
    "the fitted curve passes within 10% of every validation error used to fit it"
    assert np.all(np.abs(power(ns[:5], *params) - va_err[:5]) / va_err[:5] < 0.1)
def test_extrap():
    "the extrapolation to 1,600 trials is within 15% of the measured error"
    assert abs(power(1600, *params) - va_err[-1]) / va_err[-1] < 0.15

test_fit()
test_extrap()
print("All checks passed")

### Exercise: Linear against nearest-neighbour decoding

Write knn_decode(X_tr, s_tr, X_te, k) with Euclidean distances on standardised counts. Compare its validation MSE (k = 15) with the linear decoder's at 25, 100, 400 and 1,600 trials, averaged over 5 training sets, using 1,000 validation trials.

In [ ]:
def knn_decode(X_tr, s_tr, X_te, k):
    mu, sd = X_tr.mean(0), X_tr.std(0) + 1e-6
    A, B = (X_tr - mu) / sd, (X_te - mu) / sd
    d = (B ** 2).sum(1)[:, None] - 2 * B @ A.T + (A ** 2).sum(1)[None, :]
    nearest = np.argsort(d, axis=1)[:, :k]
    return ...

sizes = [25, 100, 400, 1600]
s_v, X_v = s_va[:1000], X_va[:1000]
lin, knn = np.zeros(4), np.zeros(4)
for k_, n in enumerate(sizes):
    l_, k_n = [], []
    for rep in range(5):
        s_tr, X_tr = simulate(n, r)
        l_.append(np.mean((decoder_fit(X_tr, s_tr)(X_v) - s_v) ** 2))
        k_n.append(np.mean((knn_decode(X_tr, s_tr, X_v, 15) - s_v) ** 2))
    lin[k_], knn[k_] = np.mean(l_), np.mean(k_n)
    print(f"n = {n:4d}: linear {lin[k_]:.4f}, nearest neighbours {knn[k_]:.4f}")

In [ ]:
import numpy as np
def test_knn():
    "knn_decode averages the stimuli of the k nearest training trials"
    Xa = np.array([[0.0], [1.0], [2.0], [10.0]]); sa = np.array([0.0, 1.0, 2.0, 3.0])
    assert np.allclose(knn_decode(Xa, sa, np.array([[0.9]]), 2), [0.5])
def test_cross():
    "the linear decoder wins with 25 trials and the nearest-neighbour decoder wins with 1,600"
    assert lin[0] < knn[0] and knn[-1] < lin[-1]

test_knn()
test_cross()
print("All checks passed")